# Titanic Survival Prediction

Predicting passenger survival (classification) and ticket fare (regression)
with scikit-learn, using the Kaggle "Titanic - Machine Learning from Disaster" data.

**Workflow:** load → inspect → EDA → clean → encode → train → evaluate → predict

**Tools:** pandas, NumPy, matplotlib, seaborn, plotly, scikit-learn

### Imports and settings

In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_curve, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score,
)

In [ ]:
RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
# Folder for charts, so you can reuse them in your README
os.makedirs("images", exist_ok=True)

In [ ]:
def save_fig(name):
    plt.savefig(f"images/{name}.png", dpi=150, bbox_inches="tight")

### Load the data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = "/content/drive/MyDrive/titanic"

In [ ]:
train = pd.read_csv(DATA_DIR + "/train.csv")
test = pd.read_csv(DATA_DIR + "/test.csv")
sample_submission = pd.read_csv(DATA_DIR + "/gender_submission.csv")

In [ ]:
print("train:", train.shape, "| test:", test.shape)
print("Only in train:", set(train.columns) - set(test.columns))

In [ ]:
train.head()

## Data dictionary

| Column | Meaning |
|---|---|
| Survived | 0 = died, 1 = survived (target) |
| Pclass | Ticket class: 1 = upper, 2 = middle, 3 = lower |
| Name | Passenger name (contains a title such as Mr, Mrs, Miss) |
| Sex | male / female |
| Age | Age in years |
| SibSp | Number of siblings or spouses aboard |
| Parch | Number of parents or children aboard |
| Ticket | Ticket number |
| Fare | Price paid |
| Cabin | Cabin number |
| Embarked | Port of embarkation: C = Cherbourg, Q = Queenstown, S = Southampton |

### Inspect the data and missing values

In [ ]:
train.info()
print()
print(train.describe())

In [ ]:
missing = pd.DataFrame({
    "train_missing": train.isnull().sum(),
    "train_%": (train.isnull().mean() * 100).round(1),
    "test_missing": test.isnull().sum().reindex(train.columns),
})
print()
print(missing[(missing["train_missing"] > 0) | (missing["test_missing"] > 0)])

In [ ]:
plt.figure(figsize=(9, 3))
sns.heatmap(train.isnull(), cbar=False, yticklabels=False, cmap="viridis")
plt.title("Missing values in train.csv (yellow = missing)")
save_fig("missing_values")
plt.show()

### Exploratory data analysis

In [ ]:
# 1. Target balance
train["Survived"].value_counts().plot(kind="bar", color=["tomato", "seagreen"])
plt.title("Survived (0 = No, 1 = Yes)")
plt.xticks(rotation=0)
plt.ylabel("Passengers")
save_fig("target_balance")
plt.show()
print("Overall survival rate:", round(train["Survived"].mean(), 3))

In [ ]:
# 2. Survival by sex and class (seaborn)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.countplot(data=train, x="Sex", hue="Survived", ax=axes[0])
axes[0].set_title("Survival by sex")
sns.countplot(data=train, x="Pclass", hue="Survived", ax=axes[1])
axes[1].set_title("Survival by class")
plt.tight_layout()
save_fig("survival_sex_class")
plt.show()

In [ ]:
# 3. Age and fare distributions (matplotlib + seaborn)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(train["Age"].dropna(), bins=30, color="steelblue", edgecolor="black")
axes[0].set_title("Age distribution")
sns.histplot(train["Fare"], bins=40, kde=True, ax=axes[1])
axes[1].set_title("Fare distribution (skewed)")
plt.tight_layout()
save_fig("age_fare_distributions")
plt.show()

In [ ]:
# 4. Age vs survival
sns.boxplot(data=train, x="Survived", y="Age")
plt.title("Age by survival")
save_fig("age_vs_survival")
plt.show()

In [ ]:
# 5. Correlation heatmap
sns.heatmap(train.select_dtypes(include=np.number).corr(),
            annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation heatmap")
save_fig("correlation_heatmap")
plt.show()

In [ ]:
# 6. Interactive plotly charts
rates = train.groupby(["Pclass", "Sex"])["Survived"].mean().reset_index()
fig = px.bar(rates, x="Pclass", y="Survived", color="Sex", barmode="group",
             title="Survival rate by class and sex")
fig.show()

fig = px.scatter(train, x="Age", y="Fare", color=train["Survived"].astype(str),
                 hover_data=["Sex", "Pclass"],
                 title="Age vs Fare (colored by survival)")
fig.show()

## Key findings from EDA

- Only about 38% of passengers survived, so "always predict died" already scores about 61%.
  Any useful model has to beat that baseline.
- Sex is the strongest signal: most women survived, most men did not.
- Passenger class matters: first class survived at a much higher rate than third.
- Age has about 20% missing values and Cabin about 77%, so cleaning decisions are needed.
- Fare is heavily right-skewed, which will make fare prediction harder for a linear model.

### One reusable preprocessing function

In [ ]:
def preprocess(train_df, test_df, engineer=False):
    """
    Clean and encode train and test the same way.
    engineer=True adds Title, FamilySize and IsAlone features.
    Returns: X, y, X_test, test_ids
    """
    train_df = train_df.copy()
    test_df = test_df.copy()
    test_ids = test_df["PassengerId"]

    if engineer:
        for df in (train_df, test_df):
            df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
            df["Title"] = df["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
            common = ["Mr", "Mrs", "Miss", "Master"]
            df["Title"] = df["Title"].where(df["Title"].isin(common), "Rare")
            df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
            df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

    # Drop columns a basic model can't use or that are mostly empty
    drop_cols = ["Cabin", "PassengerId", "Name", "Ticket"]
    train_df = train_df.drop(columns=drop_cols)
    test_df = test_df.drop(columns=drop_cols)

    # Missing values
    train_df = train_df.dropna(subset=["Embarked"])
    age_mean = train_df["Age"].mean()
    fare_mean = train_df["Fare"].mean()
    train_df["Age"] = train_df["Age"].fillna(age_mean)
    test_df["Age"] = test_df["Age"].fillna(age_mean)
    test_df["Fare"] = test_df["Fare"].fillna(fare_mean)

    # One-hot encoding
    cat_cols = ["Sex", "Embarked"] + (["Title"] if engineer else [])
    train_df = pd.get_dummies(train_df, columns=cat_cols, drop_first=True, dtype=int)
    test_df = pd.get_dummies(test_df, columns=cat_cols, drop_first=True, dtype=int)

    X = train_df.drop(columns=["Survived"])
    y = train_df["Survived"]
    X_test = test_df.reindex(columns=X.columns, fill_value=0)
    return X, y, X_test, test_ids

In [ ]:
# Version 1: the plain pipeline you asked for
X, y, X_test, test_ids = preprocess(train, test, engineer=False)
print("Plain features:", list(X.columns))
print("Missing left:", X.isnull().sum().sum(), "|", X_test.isnull().sum().sum())

### Logistic regression (classification, default hyperparameters)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

baseline_acc = (y_val == 0).mean()
print("Baseline accuracy (always predict 'died'):", round(baseline_acc, 4))

In [ ]:
log_model = LogisticRegression()          # default hyperparameters
log_model.fit(X_train, y_train)
val_pred = log_model.predict(X_val)

acc = accuracy_score(y_val, val_pred)
print("Validation accuracy:", round(acc, 4))
print()
print(classification_report(y_val, val_pred, target_names=["Died", "Survived"]))

In [ ]:
cm = confusion_matrix(y_val, val_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Died", "Survived"], yticklabels=["Died", "Survived"])
plt.title("Confusion matrix: Logistic Regression")
plt.xlabel("Predicted")
plt.ylabel("Actual")
save_fig("confusion_matrix")
plt.show()

 What did the model learn?

In [ ]:
coefs = pd.DataFrame({"feature": X.columns, "coefficient": log_model.coef_[0]})
coefs = coefs.sort_values("coefficient")

plt.figure(figsize=(7, 4))
sns.barplot(data=coefs, x="coefficient", y="feature", palette="vlag")
plt.title("Logistic regression coefficients")
plt.axvline(0, color="black", linewidth=0.8)
save_fig("coefficients")
plt.show()

In [ ]:
# ROC curve: how well does the model rank survivors above non-survivors?
probs = log_model.predict_proba(X_val)[:, 1]
fpr, tpr, _ = roc_curve(y_val, probs)
auc = roc_auc_score(y_val, probs)

plt.plot(fpr, tpr, label=f"Logistic Regression (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random guessing")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curve")
plt.legend()
save_fig("roc_curve")
plt.show()

Does feature engineering help? (cross-validation)

In [ ]:
# Version 2: add Title, FamilySize, IsAlone
X2, y2, X_test2, test_ids2 = preprocess(train, test, engineer=True)

cv_plain = cross_val_score(LogisticRegression(), X, y, cv=5, scoring="accuracy")
cv_eng = cross_val_score(LogisticRegression(), X2, y2, cv=5, scoring="accuracy")

In [ ]:
results = pd.DataFrame({
    "Feature set": ["Plain features", "With engineered features"],
    "Mean CV accuracy": [cv_plain.mean(), cv_eng.mean()],
    "Std": [cv_plain.std(), cv_eng.std()],
}).round(4)
print(results)

In [ ]:
fig = px.bar(results, x="Feature set", y="Mean CV accuracy", error_y="Std",
             range_y=[0.7, 0.9], title="5-fold cross-validated accuracy")
fig.show()

### Linear regression (predict Fare, default hyperparameters)

In [ ]:
X_reg = X.drop(columns=["Fare"])
y_reg = X["Fare"]

Xr_train, Xr_val, yr_train, yr_val = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=RANDOM_STATE
)

In [ ]:
lin_model = LinearRegression()
lin_model.fit(Xr_train, yr_train)
fare_pred = lin_model.predict(Xr_val)

print("R2:  ", round(r2_score(yr_val, fare_pred), 4))
print("MAE: ", round(mean_absolute_error(yr_val, fare_pred), 4))
print("RMSE:", round(np.sqrt(mean_squared_error(yr_val, fare_pred)), 4))

In [ ]:
fig = px.scatter(x=yr_val, y=fare_pred,
                 labels={"x": "Actual fare", "y": "Predicted fare"},
                 title="Linear regression: actual vs predicted fare")
fig.add_shape(type="line", x0=yr_val.min(), y0=yr_val.min(),
              x1=yr_val.max(), y1=yr_val.max(),
              line=dict(color="red", dash="dash"))
fig.show()

### Final predictions on test.csv

In [ ]:
# Pick the feature set that did better in Cell 11. Here I use the engineered one.
final_model = LogisticRegression()
final_model.fit(X2, y2)
test_pred = final_model.predict(X_test2)

my_submission = pd.DataFrame({"PassengerId": test_ids2, "Survived": test_pred})

In [ ]:
print(my_submission.head())
print("Same columns as sample:", list(my_submission.columns) == list(sample_submission.columns))
print("Same IDs in same order:",
      (my_submission["PassengerId"] == sample_submission["PassengerId"]).all())
print("Predicted survival rate in test:", round(my_submission["Survived"].mean(), 3))

## Results

| Model | Task | Metric | Score |
|---|---|---|---|
| Baseline (always "died") | Classification | Accuracy | 0.618 |
| Logistic Regression (plain features) | Classification | 5-fold CV accuracy | 0.7874 +/- 0.017 |
| Logistic Regression (+ engineered features) | Classification | 5-fold CV accuracy | 0.8144 +/-0.016 |
| Linear Regression | Fare prediction | R² | 0.3468 |

## Limitations and next steps

- All models use default hyperparameters by design; tuning with GridSearchCV is a next step.
- Age was filled with the global mean. Filling by title or class would likely be better.
- Fare is skewed, so a log transform should help the regression.
- Only linear models were used; tree-based models (Random Forest) could be compared.

In [ ]:
print("=== NUMBERS FOR YOUR README ===")
print("Baseline accuracy (always 'died'):", round(baseline_acc, 3))
print("Logistic Regression, validation accuracy:", round(acc, 3))
print("Logistic Regression, ROC AUC:", round(auc, 3))
print("CV accuracy, plain features:       ", round(cv_plain.mean(), 3), "+/-", round(cv_plain.std(), 3))
print("CV accuracy, engineered features:  ", round(cv_eng.mean(), 3), "+/-", round(cv_eng.std(), 3))
print("Linear Regression R2:", round(r2_score(yr_val, fare_pred), 3))
print("Linear Regression MAE:", round(mean_absolute_error(yr_val, fare_pred), 3))
print("Linear Regression RMSE:", round(np.sqrt(mean_squared_error(yr_val, fare_pred)), 3))
